# HW5: Image Captioning
---

This is the Notebook that goes with **Homework 5: Image Captioning**! 

In this notebook, you can visualize the self-attention layer in your TransformerDecoder, and generate captions using both of your models for images in the test dataset. 

This notebook can be ported to Colab very quickly, so please feel free to try that out!

This block of code imports the classes you completed in your assignment, along with additional libraries needed for the visualizations.

Feel free to add autoimport queries as needed. This notebook's code will not be auto-ran by the autograder (only the outputs will be looked at during manual grading), so do what you need to here. 

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch
import torch.nn.functional as F

from model import ImageCaptionModel
from decoder import TransformerDecoder, RNNDecoder

## Exploring the Dataset

This assignment uses the Flickr 8k dataset! Let's go ahead and pull that in!

In [ ]:
## Before this, download the dataset and run preprocessing.py as instructed. 
## This may take like 10 mins, but should only happen once so ok.
## https://www.kaggle.com/datasets/adityajn105/flickr8k?resource=download

with open('../data/data.p', 'rb') as data_file:
    data_dict = pickle.load(data_file)

# As mentioned in the handout, this assignment has 5 captions per image. This block of code 
# expands the image_feature lists to have 5 copies of each image to correspond to each of their captions 
feat_prep = lambda x: np.repeat(np.array(x).reshape(-1, 2048), 5, axis=0)
img_prep  = lambda x: np.repeat(x/255., 5, axis=0).astype(np.float64)

## Captions; preprocessed sentences with 20 window size
train_captions  = np.array(data_dict['train_captions'],dtype=np.int64);            print('train_captions:  ', train_captions.shape)
test_captions   = np.array(data_dict['test_captions'],dtype=np.int64);             print('test_captions:   ', test_captions.shape)

## 2048-D resnet embeddings of images.
train_img_feats = feat_prep(data_dict['train_image_features']);     print('\ntrain_img_feats: ', train_img_feats.shape)
test_img_feats  = feat_prep(data_dict['test_image_features']);      print('test_img_feats:  ', test_img_feats.shape)

## Small subset of actual images for visualization purposes. 
## These are just for the first 100 images of each (clones 5 times)
train_images    = img_prep(data_dict['train_images']);              print('\ntrain_images:    ', train_images.shape)
test_images     = img_prep(data_dict['test_images']);               print('test_images:     ', test_images.shape)

## Conversion dictionaries to go between word and label index
word2idx        = data_dict['word2idx']
idx2word        = data_dict['idx2word']

Since the images take up a lot of data, we only kept a sliver of the original images. Feel free to update the preprocessing to retain all of the images if you'd like. Below is a visualization of some of the data:

In [ ]:

for i in range(5):
    for j in range(5):
        print(f'Caption {j+1}:', ' '.join([idx2word[idx] for idx in train_captions[i * 5 + j]]))
    plt.imshow(train_images[i * 5])
    plt.show()

## Visualization

After training our Transformer model, you can visualize the self-attention layer to examine the behavior of your attention heads and see if any patterns emerge. 

To test out the components of the model interactively, you'll need to deconstruct selections of the model/runner code and get an instance of the model in an interactive context (aka inside the notebook). 



In [ ]:
## Feel free to insert auto-reloads as necessary
from assignment import parse_args, load_model
from decoder import TransformerDecoder, RNNDecoder

## TODO: Pull your model into the notebook. This is heavily based off of assignment.py,
##   and feel free to reuse as much as you want.
##   Update the chkpt_path values to match wherever you saved your checkpoints.

args = parse_args(['--type', 'rnn', '--task', 'both', '--data', '../data/data.p'])

args.chkpt_path = '../transformer_model'
tra_imcap = load_model(args)

args.chkpt_path = '../rnn_model'
rnn_imcap = load_model(args)

In [ ]:
## PyTorch model architecture + parameter count (replaces Keras .summary())
print(rnn_imcap)
print()
total_params = sum(p.numel() for p in rnn_imcap.parameters())
print(f"RNN model – total parameters: {total_params:,}")

In [ ]:
print(tra_imcap)
print()
total_params = sum(p.numel() for p in tra_imcap.parameters())
print(f"Transformer model – total parameters: {total_params:,}")

Now that we have our model, we want to peek inside the `TransformerBlock` to visualise the attention weights it computed.

In PyTorch we can access sub-modules and weights directly via attribute access — no need for a Functional API.  We will manually run the same projections the block uses internally and call `AttentionMatrix` to get the weight matrices.

The cell below visualises two attention matrices for a randomly chosen test image:

1. **Self-attention** – how each caption token attends to every other caption token (masked causal attention).  
2. **Cross-attention** – how each caption token attends to the image context (a single encoded image token).

Move through the heatmaps to see which words the model focuses on as it builds the caption.

In [ ]:
import numpy as np

index = np.random.choice(np.array(list(range(0, 500, 5))))

caption    = test_captions[index][:-1]        # drop last <pad> / <end>
image_feat = test_img_feats[index]
image      = test_images[index]

print("Image number:", index)


def get_attention(tra_imcap, image_feat_np, caption_np):
    """
    Extracts self-attention and cross-attention matrices from the
    TransformerDecoder using PyTorch.

    TODO: If you implemented MultiHeadedAttention, update this function
          to loop over each head and visualise them separately, then also
          show the average across heads.

    Returns
    -------
    self_atten         : Tensor [1, T, T]  – caption-to-caption attention
    self_context_atten : Tensor [1, T, 1]  – caption-to-image attention
    """
    tra_imcap.eval()
    with torch.no_grad():
        # --- prepare inputs -------------------------------------------------
        img_tensor = torch.FloatTensor(image_feat_np).unsqueeze(0)   # [1, 2048]
        cap_tensor = torch.LongTensor(caption_np).unsqueeze(0)       # [1, T]

        # --- replicate what TransformerDecoder.forward() does ---------------
        # 1. Project image to hidden_size and add sequence dim
        encoded_images = tra_imcap.decoder.image_embedding(img_tensor)   # [1, H]
        encoded_images = encoded_images.unsqueeze(1)                      # [1, 1, H]

        # 2. Embed + positional-encode the caption tokens
        captions_emb = tra_imcap.decoder.encoding(cap_tensor)            # [1, T, H]

        # --- extract self-attention (masked) --------------------------------
        sa_head = tra_imcap.decoder.decoder.self_atten
        K_sa = sa_head.K(captions_emb)   # [1, T, out]
        Q_sa = sa_head.Q(captions_emb)   # [1, T, out]
        self_atten = sa_head.attn_mtx(K_sa, Q_sa)   # [1, T, T]

        # --- extract cross-attention (image context) ------------------------
        ca_head = tra_imcap.decoder.decoder.self_context_atten
        K_ca = ca_head.K(encoded_images)   # [1, 1, out]
        Q_ca = ca_head.Q(captions_emb)     # [1, T, out]
        self_context_atten = ca_head.attn_mtx(K_ca, Q_ca)   # [1, T, 1]

    return self_atten, self_context_atten


def vis_attention(atten_mtx, caption_np, idx2word, title="Attention"):
    """
    Visualises an attention matrix with matplotlib.

    atten_mtx : Tensor [1, Q, K]
    """
    caption_words = [idx2word[idx] for idx in caption_np]
    end_idx = (caption_words.index('<end>')
               if '<end>' in caption_words else len(caption_words))
    caption_words = caption_words[:end_idx]
    T = end_idx

    attn = atten_mtx[0, :T, :].detach().numpy()   # [T, K]
    K_len = attn.shape[1]

    fig, ax = plt.subplots(figsize=(max(5, K_len), max(5, T // 2 + 1)))
    im = ax.imshow(attn, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)

    ax.set_yticks(range(T))
    ax.set_yticklabels(caption_words, fontsize=9)

    if K_len == T:
        # square self-attention matrix → label both axes with words
        ax.set_xticks(range(T))
        ax.set_xticklabels(caption_words, rotation=45, ha='right', fontsize=9)
        ax.set_xlabel("Keys (caption)")
        ax.set_ylabel("Queries (caption)")
    else:
        # context attention: K_len == 1 → label x-axis as "image"
        ax.set_xticks([0])
        ax.set_xticklabels(["<image>"])
        ax.set_xlabel("Context key")
        ax.set_ylabel("Caption query")

    ax.set_title(title)
    plt.tight_layout()
    plt.show()


self_atten, self_context_atten = get_attention(tra_imcap, image_feat, caption)

print("Self-attention (caption → caption)")
vis_attention(self_atten, caption, idx2word, title="Self-Attention")

print("\nCross-attention (caption → image)")
vis_attention(self_context_atten, caption, idx2word, title="Cross-Attention (image context)")

plt.imshow(image)
plt.title(f"Image #{index}")
plt.axis('off')
plt.show()

### Caption Generation
Now that you have trained both of your models, it's time to use them to generate original captions for images in the testing set. First, the model is given the <start\> token and asked to generate probabilites for the next word in the sequence. The next token is chosen by sampling from that probability. This process repeats until the model generates the <end\> token, or the maximum sequence length is reached.

 



There is still one piece of this equation missing. The tokens are sampled from the probabilities your models generate, but your models were required to output logits, not probabilities. This is becasue this assignment, like many NLP models, uses temperature as a parameter in text generation. If the models sampled from  probabilies calculated by simply applying softmax to the logits, then the probability of the most likely word will usually be very high and the models will usually genrate the same, most probable caption every time. We use the temperature as a parameter to even out the probabilites so the model produces more 'creative' captions. This is done by dividing the logits by the temperature parameter before applying softmax. Higher temprature values will give a more creative captiong, while temprature values closer to 0 will be more greedy. Check out [this](https://lukesalamone.github.io/posts/what-is-temperature/) article for a demonstration and further explaination of temprature in NLP models.


The following blocks of code will generate a caption for the image currently selected for the attention visualization above. Try playing around with different temperature values and see how it changes the captions your models generate

In [ ]:
def gen_caption_temperature(model, image_embedding, wordToIds, padID, temp, window_length):
    """
    Generates a caption from an ImageCaptionModel using temperature-scaled sampling.

    Parameters
    ----------
    model           : ImageCaptionModel (PyTorch)
    image_embedding : np.ndarray [2048]
    wordToIds       : dict  word → int
    padID           : int   padding token id
    temp            : float sampling temperature  (lower = greedier)
    window_length   : int   maximum caption length
    """
    idsToWords  = {v: k for k, v in wordToIds.items()}
    unk_token   = wordToIds['<unk>']
    caption_so_far = [wordToIds['<start>']]

    model.eval()
    with torch.no_grad():
        while (len(caption_so_far) < window_length
               and caption_so_far[-1] != wordToIds['<end>']):
            # Pad caption to window_length
            pad_len = window_length - len(caption_so_far)
            caption_input = caption_so_far + [padID] * pad_len

            img_tensor = torch.FloatTensor(image_embedding).unsqueeze(0)  # [1, 2048]
            cap_tensor = torch.LongTensor(caption_input).unsqueeze(0)     # [1, W]

            logits = model(img_tensor, cap_tensor)                        # [1, W, V]
            logits = logits[0, len(caption_so_far) - 1]                   # [V]

            # Temperature scaling then softmax
            probs = F.softmax(logits / temp, dim=-1).numpy()

            next_token = unk_token
            attempts   = 0
            while next_token == unk_token and attempts < 5:
                next_token = int(np.random.choice(len(probs), p=probs))
                attempts  += 1

            caption_so_far.append(next_token)

    return ' '.join([idsToWords[x] for x in caption_so_far][1:-1])


temperature = 0.05
print(gen_caption_temperature(
    tra_imcap, image_feat, word2idx, word2idx['<pad>'], temperature, args.window_size
))

In [ ]:
temperature = 0.2
print(gen_caption_temperature(
    tra_imcap, image_feat, word2idx, word2idx['<pad>'], temperature, args.window_size
))

**NOTE:** You may want to try a different image. Sometimes you get really unlucky with random selection.

## Generating Sentences for Training Data 

In [ ]:
temperature = 0.05
indices = np.random.choice(np.array(list(range(0, 500, 5))), 10, replace=False)
for i in indices:
    curr_image_feat = train_img_feats[i]
    curr_image      = train_images[i]
    for j in range(5):   ## Display all 5 captions this image was trained on
        words = [idx2word[x] for x in train_captions[i + j][:-1]
                 if idx2word[x] not in ('<pad>', '<start>', '<end>')]
        print(f'C{j+1}:', ' '.join(words))
    print('RNN:', gen_caption_temperature(rnn_imcap, curr_image_feat, word2idx, word2idx['<pad>'], temperature, args.window_size))
    print('TRA:', gen_caption_temperature(tra_imcap, curr_image_feat, word2idx, word2idx['<pad>'], temperature, args.window_size))
    plt.imshow(curr_image)
    plt.axis('off')
    plt.show()

### Trying out on things in testing set!

In [ ]:
temperature = 0.05
indices = np.random.choice(np.array(list(range(0, 500, 5))), 10, replace=False)
for i in indices:
    curr_image_feat = test_img_feats[i]
    curr_image      = test_images[i]
    for j in range(5):
        words = [idx2word[x] for x in test_captions[i + j][:-1]
                 if idx2word[x] not in ('<pad>', '<start>', '<end>')]
        print(f'C{j+1}:', ' '.join(words))
    print('RNN:', gen_caption_temperature(rnn_imcap, curr_image_feat, word2idx, word2idx['<pad>'], temperature, args.window_size))
    print('TRA:', gen_caption_temperature(tra_imcap, curr_image_feat, word2idx, word2idx['<pad>'], temperature, args.window_size))
    plt.imshow(curr_image)
    plt.axis('off')
    plt.show()

# Conclusion!
Congrats! You have finished this assignment! Below, put down your favorite captions that your RNN and Transformer models both generated!  

In [ ]:
## TODO: Fill in the image indices below and run this cell to display your
##       favourite captions alongside the images.

rnn_image_index = 0   # replace with your chosen index
rnn_caption = gen_caption_temperature(
    rnn_imcap, test_img_feats[rnn_image_index],
    word2idx, word2idx['<pad>'], temperature, args.window_size
)

tra_image_index = 0   # replace with your chosen index
tra_caption = gen_caption_temperature(
    tra_imcap, test_img_feats[tra_image_index],
    word2idx, word2idx['<pad>'], temperature, args.window_size
)

print("RNN caption:", rnn_caption)
plt.imshow(test_images[rnn_image_index])
plt.axis('off')
plt.show()

print("Transformer caption:", tra_caption)
plt.imshow(test_images[tra_image_index])
plt.axis('off')
plt.show()